## dataset

#### 该数据集包含2892张图片，图片内容为白色背景下做出经典石头、剪刀、布手势的各类手部图像。这些图像通过计算机图形技术生成，因此这也是一个合成数据集。每张图片尺寸为300×300像素，拥有四个通道（RGBA）。
#### RGBA代表红-绿-蓝-阿尔法，它是传统的RGB色彩模型外加一个阿尔法通道，用于表示每个像素的不透明度。可以暂时不用管阿尔法通道，后续会将其移除。

## Example

![](../pictures/rps.png)

## ImageFolder
这本身不是一个数据集，而是一个通用数据集模板。你可以搭配自己的图像使用，前提是将图像妥善整理到各个子文件夹中，每个子文件夹以类别命名，并存放对应的图像。

石头剪刀布数据集就是按这种方式组织的：在训练集的文件夹内，有三个子文件夹，分别对应三个类别（石头、布、剪刀）。

ImageFolder数据集只需要根文件夹，在本例中就是rps/train文件夹。不过它还可以接收另外四个可选参数：
- transform：这个你已经了解；它指定数据集对每张图像执行哪些变换，例如我们在前几章见过的数据增强变换。
- target_transform：到目前为止，我们的目标标签一直都是整数，所以这个参数暂时用不上；如果你的目标本身也是图像（例如分割任务），这个参数就会发挥作用。
- loader：用于从指定路径加载图像的函数，适用于使用PIL无法处理的特殊或非标准图像格式的场景。
- is_valid_file：用于检查文件是否损坏的函数。

In [ ]:
import numpy as np
import torch
from torch import nn, optim
from torch.nn import functional as F
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
from torchvision.transforms import Compose, Normalize, Resize, ToTensor

In [ ]:
# temp dataset
temp_transform = Compose([Resize(28), ToTensor()])
temp_dataset = ImageFolder(root="../Datasets/RPS/train", transform=temp_transform)

In [ ]:
temp_dataset[0][0].shape, temp_dataset[0][1]

In [ ]:
# No need to bother with shuffling, as this is not the data loader we’ll use to train the
# model anyway. We’ll use it to compute statistics only. By the way, we need
# statistics for each channel, as required by the Normalize() transform.
bs = 64
temp_loader = DataLoader(temp_dataset, batch_size=bs)

In [ ]:
images, labels = next(iter(temp_loader))
# NCHW
n_samples, n_channels, n_height, n_weight = images.size()
# Flatten HW into a single dimension
flatten_per_channel = images.reshape(n_samples, n_channels, -1)

In [ ]:
# Computes statistics of each image per channel
# Average pixel value per channel
# (n_samples, n_channels)
means = flatten_per_channel.mean(dim=2)
# Standard deviation of pixel values per channel
# (n_samples, n_channels)
stds = flatten_per_channel.std(dim=2)

In [ ]:
means.shape, stds.shape

In [ ]:
# Adds up statistics of all images in a mini-batch
# (1, n_channels)
sum_means = means.sum(dim=0)
sum_stds = stds.sum(dim=0)

In [ ]:
n_samples = torch.tensor([n_samples] * n_channels).float()

In [ ]:
torch.stack([n_samples, sum_means, sum_stds])

In [ ]:
for x_batch, y_batch in temp_loader:
    print(x_batch.shape, y_batch.shape)

In [ ]:
# statistics of each image per channel
results = []
for images, labels in temp_loader:
    # NCHW
    n_samples, n_channels, n_height, n_weight = images.size()
    # Flatten HW into a single dimension
    flatten_per_channel = images.reshape(n_samples, n_channels, -1)
    # Computes statistics of each image per channel
    # Average pixel value per channel
    # (n_samples, n_channels)
    means = flatten_per_channel.mean(dim=2)
    # Standard deviation of pixel values per channel
    # (n_samples, n_channels)
    stds = flatten_per_channel.std(dim=2)
    n_samples = torch.tensor([n_samples] * n_channels).float()
    # Adds up statistics of all images in a mini-batch
    # (1, n_channels)
    sum_means = means.sum(dim=0)
    sum_stds = stds.sum(dim=0)
    result = torch.stack([n_samples, sum_means, sum_stds])
    # Append results
    results.append(result)
results = torch.stack(results).sum(dim=0)


In [ ]:
total_samples, total_means, total_stds = results
norm_mean = total_means / total_samples
norm_std = total_stds / total_samples

In [ ]:
norm_mean, norm_std

## Real Dataset

In [ ]:
size = 64
composer = Compose([Resize(size), ToTensor(), Normalize(norm_mean, norm_std)])
train_data = ImageFolder(root="../Datasets/RPS/train", transform=composer)
val_data = ImageFolder(root="../Datasets/RPS/test", transform=composer)
# Builds a loader of each set
bs = 64
train_loader = DataLoader(train_data, batch_size=bs, shuffle=True)
val_loader = DataLoader(val_data, batch_size=bs)

## model

In [ ]:
class RPScnn(nn.Module):
    def __init__(self, n_filters, p=0.0):
        super().__init__()
        self.n_filters = n_filters
        self.p = p
        # Creates the convolution layers
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=n_filters, kernel_size=3)
        self.conv2 = nn.Conv2d(
            in_channels=n_filters, out_channels=n_filters * 2, kernel_size=3
        )
        self.conv3 = nn.Conv2d(
            in_channels=n_filters * 2, out_channels=n_filters * 4, kernel_size=3
        )
        # Creates the linear layers
        self.fc1 = nn.Linear(n_filters * 4 * 6 * 6, 50)
        self.fc2 = nn.Linear(50, 3)
        # Creates dropout layers
        self.drop = nn.Dropout(self.p)

    def featurizer(self, x):
        # First convolutional block
        # 3@64x64 -> n_filters@62x62 -> n_filters@31x31
        x = self.conv1(x)
        x = F.relu(x)
        x = F.max_pool2d(x, kernel_size=2)
        # Second convolutional block
        # n_filters@31x31 -> n_filters@29x29 -> n_filters@14x14
        x = self.conv2(x)
        x = F.relu(x)
        x = F.max_pool2d(x, kernel_size=2)
        # Third convolutional block
        # n_filters@14x14 -> n_filters@12x12 -> n_filters@6x6
        x = self.conv3(x)
        x = F.relu(x)
        x = F.max_pool2d(x, kernel_size=2)
        # Input dimension (n_filters*4@6x6)
        # Output dimension (n_filters * 4 * 6 * 6)
        x = nn.Flatten()(x)
        return x

    def classifier(self, x):
        # Classifier
        # Hidden Layer
        # Input dimension (n_feature * 4 * 6 * 6)
        # Output dimension (50)
        if self.p > 0:
            x = self.drop(x)
        x = self.fc1(x)
        x = F.relu(x)
        # Output Layer
        # Input dimension (50)
        # Output dimension (3)
        if self.p > 0:
            x = self.drop(x)
        x = self.fc2(x)
        return x

    def forward(self, x):
        x = self.featurizer(x)
        x = self.classifier(x)
        return x

In [ ]:
def mini_batch_train(data_loader, model, optimizer, loss_fn):
    mini_batch_losses = []
    for x_batch, y_batch in data_loader:
        # 置为训练状态
        model.train()
        # Step 1 - 前向计算预测值
        yhat = model(x_batch)
        # Step 2 - 计算损失
        mini_batch_loss = loss_fn(yhat, y_batch)
        # Step 3 - 计算梯度
        mini_batch_loss.backward()
        # Step 4 - 参数更新
        optimizer.step()
        optimizer.zero_grad()

        mini_batch_losses.append(mini_batch_loss.item())

    loss = np.mean(mini_batch_losses)
    return loss

In [ ]:
def mini_batch_val(data_loader, model, loss_fn):
    mini_batch_losses = []
    for x_batch, y_batch in data_loader:
        # 置为验证状态
        model.eval()
        # Step 1 - 前向计算预测值
        yhat = model(x_batch)
        # Step 2 - 计算损失
        mini_batch_loss = loss_fn(yhat, y_batch)
        mini_batch_losses.append(mini_batch_loss.item())

    loss = np.mean(mini_batch_losses)
    return loss

In [ ]:
def train(model, train_loader, test_loader, loss_fn, optimizer, epochs):
    # 指定随机数种子，可以再现数据
    np.random.seed(23)
    torch.manual_seed(23)

    # 循环轮数计数
    total_epochs = 0

    losses = []  # 每轮训练的损失
    val_losses = []  #  每轮验证数据的损失

    for epoch in range(epochs):
        model.train()
        total_epochs += 1

        # 进入mini-batch的内循环
        loss = mini_batch_train(train_loader, model, optimizer, loss_fn)
        losses.append(loss)

        # 验证数据集（验证不需要计算梯度）
        with torch.no_grad():
            # 验证集的 mini_batch
            val_loss = mini_batch_val(test_loader, model, loss_fn)
            val_losses.append(val_loss)
        print(
            f"Epoch {total_epochs}: train loss = {loss:.4f}, val loss = {val_loss:.4f}"
        )

    return losses, val_losses

In [ ]:
model_rps = RPScnn(n_filters=5, p=0.2)
loss_fn = nn.CrossEntropyLoss()
optimizer_rps = optim.Adam(model_rps.parameters(), lr=3e-4)
epochs = 50
lr = 3e-4
losses, val_losses = train(
    model_rps, train_loader, val_loader, loss_fn, optimizer_rps, epochs
)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

plt.style.use("fivethirtyeight")


def plot_losses():
    fig = plt.figure(figsize=(10, 4))
    plt.plot(losses, label="Training Loss", c="b")
    plt.plot(val_losses, label="Validation Loss", c="r")
    plt.yscale("log")
    plt.xlabel("Epochs")
    plt.ylabel("Loss")
    plt.legend()
    plt.tight_layout()
    return fig


fig = plot_losses()

In [ ]:
import torchmetrics

accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=3)
for _, (x, y) in enumerate(val_loader):
    yhat = model_rps(x)
    accuracy(yhat, y)
accuracy.compute()
print(f"Test Accuracy: {accuracy.compute().item() * 100:.2f}%")